# 7.2 RAG / RAG

先检索再生成。手写关键词检索和向量检索，融合与重排，并在一个虚构小镇的资料上搭建完整的问答流程。

Retrieve first, then generate. Keyword and vector retrieval by hand, fusion and reranking, and a complete question-answering pipeline over material about an invented town.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写 BM25 与向量检索 / BM25 and vector retrieval by hand

实现 BM25 的索引和打分，以及用均值池化得到句向量的函数。测试先在一个可以手算的三句话语料上逐项核对 BM25 的数值，再在 30 个段落上比较两种检索方法对两类问题的表现。如果安装了 FAISS，还会检查它返回的前 3 名与手写的结果相同。

Implement the indexing and scoring of BM25 and a function that produces sentence vectors by mean pooling. The tests first check the BM25 values term by term on a three-sentence corpus that can be computed by hand, and then compare the two retrieval methods on two kinds of questions over 30 passages. If FAISS is installed, they also check that its top 3 equal the hand-written result.

需要联网下载模型。/ Downloads a model.

In [ ]:
import math
import re
from collections import Counter
import torch
from transformers import AutoModel, AutoTokenizer


def tokenize(text):
    """Lower-case words and numbers; identifiers such as lh-204 stay in one piece. Nothing to change here."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


class BM25:
    """Okapi BM25, the standard keyword ranking function.

    score(q, d) = sum over the query terms t of
        idf(t) * f(t, d) * (k1 + 1) / ( f(t, d) + k1 * (1 - b + b * |d| / avgdl) )
    idf(t) = ln( (N - n_t + 0.5) / (n_t + 0.5) + 1 )

    f(t, d): how often t occurs in d      |d|: number of tokens in d      avgdl: mean document length
    N: number of documents                n_t: number of documents that contain t
    """

    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        # TODO 1: tokenise every document; store self.docs, self.N, self.avgdl,
        #         self.tf (a Counter per document) and self.idf (a dict term -> idf)
        raise NotImplementedError

    def scores(self, query):
        """The BM25 score of every document for this query, as a list of floats.
        A query term that occurs twice in the query counts twice. Terms not in a document add nothing."""
        # TODO 2
        raise NotImplementedError


@torch.no_grad()
def embed(model, tokenizer, texts):
    """Sentence embeddings by mean pooling, as a tensor (n, d) with rows of length 1.

    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    Average model(**enc).last_hidden_state over the REAL tokens of each text (use enc["attention_mask"],
    padding must not count), then normalise every row to unit length.
    """
    # TODO 3
    raise NotImplementedError


def dense_scores(doc_vectors, query_vector):
    """Cosine similarity of every document vector (n, d) with one query vector (d,), as a list of floats."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
from transformers import AutoModel, AutoTokenizer

# 1. BM25 on a corpus small enough to check by hand
docs = ["the cat sat", "the dog sat on the mat", "cats and dogs"]
bm = BM25(docs)
assert bm.N == 3 and abs(bm.avgdl - 4.0) < 1e-9
assert abs(bm.idf["cat"] - math.log((3 - 1 + 0.5) / (1 + 0.5) + 1)) < 1e-9          # in 1 of 3 documents
assert abs(bm.idf["the"] - math.log((3 - 2 + 0.5) / (2 + 0.5) + 1)) < 1e-9          # in 2 of 3 documents
assert bm.idf["cat"] > bm.idf["the"], "a rarer term carries more weight"
s = bm.scores("the cat")
norm0 = 1.5 * (1 - 0.75 + 0.75 * 3 / 4)
expected0 = bm.idf["the"] * 1 * 2.5 / (1 + norm0) + bm.idf["cat"] * 1 * 2.5 / (1 + norm0)
norm1 = 1.5 * (1 - 0.75 + 0.75 * 6 / 4)
expected1 = bm.idf["the"] * 2 * 2.5 / (2 + norm1)
assert abs(s[0] - expected0) < 1e-9 and abs(s[1] - expected1) < 1e-9 and s[2] == 0
assert s[0] > s[1] > s[2]
assert bm.scores("zebra") == [0, 0, 0]
# term frequency saturates: ten occurrences are worth less than three times one occurrence
sat = BM25(["x " * 10 + "pad pad", "x pad pad pad pad pad pad pad pad pad pad pad", "pad"])
assert sat.scores("x")[1] < sat.scores("x")[0] < 3 * sat.scores("x")[1]

CORPUS = [
    "The Lindenhof Observatory was founded in 1887 by the astronomer Clara Wendt.",
    "The main telescope of the Lindenhof Observatory is a 62-centimetre refractor named Greta.",
    "Visitors can tour the observatory on Thursday evenings; tickets cost 9 euros.",
    "The Varnholt Institute studies river sediments and is located on Mill Street in Lindenhof.",
    "Dr. Jonas Erken has directed the Varnholt Institute since 2019.",
    "The sediment archive of the institute holds 14,300 core samples stored at four degrees Celsius.",
    "Bus line 47 connects Lindenhof station with the observatory every twenty minutes.",
    "The Lindenhof town library closes at 6 pm on weekdays and stays closed on Sundays.",
    "The annual Lantern Festival in Lindenhof takes place on the second Saturday of October.",
    "The Sorrel Bridge, built from sandstone in 1754, is the oldest crossing over the river Alme.",
    "Error code LH-204 on the Greta telescope control unit means that the dome shutter failed to open.",
    "Error code LH-311 indicates that the tracking motor has overheated and must cool for thirty minutes.",
    "The bakery Kornblume on Market Square is known for its plum cake, sold only in September.",
    "The drinking water of Lindenhof comes from the Hallern reservoir, eleven kilometres north of the town.",
    "The sample submission form of the Varnholt Institute is called form S-17 and must be sent two weeks in advance.",
    "The Alme river floods most often in March, when snow melts in the Tessin hills.",
    "The football club of the town, SV Lindenhof 1921, plays its home matches at the Brunnen stadium.",
    "The Brunnen stadium has a capacity of 3,800 spectators.",
    "Mayor Ruth Albrecht was elected in 2022 for a term of six years.",
    "The weekly market is held on Wednesdays and Saturdays from 7 am to 1 pm on Market Square.",
    "The library of the observatory keeps the notebooks of Clara Wendt, which record 412 nights of observation.",
    "Parking near the observatory is limited to 30 spaces; the lot closes at midnight.",
    "The Tessin hills reach their highest point at the Rabenkopf, 847 metres above sea level.",
    "The Lindenhof museum of local history is housed in a former salt warehouse on Quay Lane.",
    "Error code LH-207 means that the focus motor has reached its end stop.",
    "Error code LH-240 means that the weather station reports wind above the safe limit.",
    "Error code LH-312 indicates that the tracking encoder has lost its reference position.",
    "Form S-71 of the Varnholt Institute is the request for access to the cold storage room.",
    "Form S-12 is used to report damaged sample containers to the archive staff.",
    "Bus line 74 runs from Lindenhof station to the Brunnen stadium on match days only.",
]
# (question, index of the passage that answers it, a string the right answer must contain)
QUERIES = [
    ("What does error code LH-204 mean?", 10, "shutter"),
    ("What is form S-17 used for?", 14, "submission"),
    ("Since which year has Jonas Erken directed the institute?", 4, "2019"),
    ("What is the capacity of the Brunnen stadium?", 17, "3,800"),
    ("How high is the Rabenkopf?", 22, "847"),
    ("What does LH-311 indicate?", 11, "overheat"),
    ("When was the Sorrel Bridge built?", 9, "1754"),
    ("Where does bus line 47 go?", 6, "observatory"),
    ("What does LH-207 mean?", 24, "focus"),
    ("What does error LH-312 indicate?", 26, "encoder"),
    ("What is form S-71 for?", 27, "cold storage"),
    ("Where does bus line 74 go?", 29, "stadium"),
    ("Who established the place where people look at the stars?", 0, "Wendt"),
    ("How much do I pay for an evening visit to see the telescope?", 2, "9 euros"),
    ("At what temperature are the drilled mud cylinders kept?", 5, "four"),
    ("In which month is the river most likely to overflow its banks?", 15, "March"),
    ("Where does the tap water of the town originate?", 13, "Hallern"),
    ("On which days can I buy fresh vegetables from stalls in the town centre?", 19, "Wednesday"),
    ("Who leads the local government?", 18, "Albrecht"),
    ("When is the yearly celebration with lights held?", 8, "October"),
]

bm = BM25(CORPUS)
rank = lambda scores: sorted(range(len(scores)), key=lambda i: -scores[i])
assert rank(bm.scores("What does error code LH-204 mean?"))[0] == 10, "an exact identifier is what keyword search is good at"

# 2. Embeddings
NAME = "sentence-transformers/all-MiniLM-L6-v2"          # 23 million parameters, Apache 2.0
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME).eval()
D = embed(model, tok, CORPUS)
assert D.shape == (30, 384) and torch.allclose(D.norm(dim=-1), torch.ones(30), atol=1e-5)
alone = embed(model, tok, [CORPUS[17]])[0]
assert torch.allclose(alone, D[17], atol=1e-4), "padding must not change an embedding: use the attention mask when averaging"
v = embed(model, tok, ["Who established the place where people look at the stars?", "a place for watching stars", "the price of bread"])
assert (v[1] @ D[0]) > (v[2] @ D[0]) + 0.2

# 3. The two methods fail on different questions.
q = "Who established the place where people look at the stars?"       # answered by passage 0, with no word in common
r_bm25 = rank(bm.scores(q)).index(0) + 1
r_dense = rank(dense_scores(D, embed(model, tok, [q])[0])).index(0) + 1
assert r_dense == 1 and r_bm25 > 3, (r_dense, r_bm25)
q2 = "What does error code LH-204 mean?"
r2_bm25 = rank(bm.scores(q2)).index(10) + 1
r2_dense = rank(dense_scores(D, embed(model, tok, [q2])[0])).index(10) + 1
assert r2_bm25 == 1

try:                                                       # optional: the same search with FAISS, if it is installed
    import faiss
    index = faiss.IndexFlatIP(384)
    index.add(D.numpy())
    _, ids = index.search(embed(model, tok, [q]).numpy(), 3)
    assert ids[0].tolist() == rank(dense_scores(D, embed(model, tok, [q])[0]))[:3]
    note = "FAISS returns the same top 3"
except ImportError:
    note = "FAISS not installed, skipped"

print(f"m72_bm25: all tests passed ({note})")
print(f"  {q!r}: gold passage at rank {r_bm25} for BM25, rank {r_dense} for embeddings")
print(f"  {q2!r}: gold passage at rank {r2_bm25} for BM25, rank {r2_dense} for embeddings")

<details><summary><b>提示 1 / Hint 1</b></summary>

文档频率用 `Counter(term for d in self.docs for term in set(d))`，注意每个文档里同一个词只计一次。

Document frequencies come from `Counter(term for d in self.docs for term in set(d))`. Note that a word counts once per document.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

均值池化是 `(hidden * mask).sum(1) / mask.sum(1)`，其中 `mask = enc['attention_mask'][..., None].float()`。

Mean pooling is `(hidden * mask).sum(1) / mask.sum(1)` with `mask = enc['attention_mask'][..., None].float()`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

归一化用 `torch.nn.functional.normalize(mean, dim=-1)`，之后点积就是余弦相似度。

Normalize with `torch.nn.functional.normalize(mean, dim=-1)`, after which the dot product is the cosine similarity.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import re
from collections import Counter
import torch
from transformers import AutoModel, AutoTokenizer


def tokenize(text):
    """Lower-case words and numbers; identifiers such as lh-204 stay in one piece. Nothing to change here."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [tokenize(d) for d in docs]
        self.N = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / self.N
        self.tf = [Counter(d) for d in self.docs]
        df = Counter(term for d in self.docs for term in set(d))
        self.idf = {term: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for term, n in df.items()}

    def scores(self, query):
        out = []
        for doc, tf in zip(self.docs, self.tf):
            norm = self.k1 * (1 - self.b + self.b * len(doc) / self.avgdl)
            s = 0.0
            for term in tokenize(query):
                f = tf.get(term, 0)
                if f:
                    s += self.idf[term] * f * (self.k1 + 1) / (f + norm)
            out.append(s)
        return out


@torch.no_grad()
def embed(model, tokenizer, texts):
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    hidden = model(**enc).last_hidden_state                       # (n, T, d)
    mask = enc["attention_mask"][..., None].float()               # padding positions must not count
    mean = (hidden * mask).sum(1) / mask.sum(1)
    return torch.nn.functional.normalize(mean, dim=-1)


def dense_scores(doc_vectors, query_vector):
    return (doc_vectors @ query_vector).tolist()                  # cosine similarity, since all vectors have length 1
```

</details>

## 2. RRF 混合检索与 Recall@k、MRR / Hybrid retrieval with RRF, Recall@k and MRR

实现倒数排名融合和两个检索评价指标，然后在 20 个问题上比较 BM25、向量检索和两者的融合。问题分为沿用原文用词的和换了说法的两类，表格按类别分别给出结果。运行后请注意融合的总体结果是否好于其中最好的单一方法。

Implement reciprocal rank fusion and two retrieval metrics, then compare BM25, vector retrieval and their fusion on 20 questions. The questions fall into those that reuse the wording of the passage and paraphrases, and the table gives results for each kind. Note whether the overall result of fusion is better than the best single method.

需要联网下载模型。/ Downloads a model.

In [ ]:
from collections import defaultdict

# BM25, tokenize, embed and dense_scores from exercise 1 are defined above.


def ranking(scores):
    """Document indices sorted by score, best first."""
    # TODO 1
    raise NotImplementedError


def rrf(rankings, k=60):
    """Reciprocal rank fusion of several rankings (each a list of document indices, best first).

    fused_score(d) = sum over the rankings of 1 / (k + position of d), positions counted from 1.
    A document missing from a ranking gets nothing from it. Returns the fused ranking, best first.
    Only positions are used, so scores on different scales (BM25 and cosine) can be combined.
    """
    # TODO 2
    raise NotImplementedError


def recall_at_k(rankings, gold, k):
    """Fraction of queries whose gold document is among the first k results.
    rankings: one ranking per query. gold: the index of the right document for each query."""
    # TODO 3
    raise NotImplementedError


def mrr(rankings, gold):
    """Mean reciprocal rank: the average of 1 / (rank of the gold document), 0 if it is missing."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModel, AutoTokenizer

assert ranking([0.1, 0.9, 0.5]) == [1, 2, 0]
# two rankings that disagree on first place: the document both place high wins
assert rrf([[0, 1, 2], [1, 2, 0]]) == [1, 0, 2]
assert rrf([[3, 1], [1]], k=60)[0] == 1 and set(rrf([[3, 1], [1]])) == {1, 3}
a = rrf([[0, 1, 2, 3], [3, 2, 1, 0]], k=1)
assert set(a[:2]) == {0, 3}, "with a small k the top positions dominate"
assert recall_at_k([[2, 0, 1], [1, 2, 0]], [0, 0], 1) == 0 and recall_at_k([[2, 0, 1], [1, 2, 0]], [0, 0], 2) == 0.5
assert abs(mrr([[2, 0, 1], [1, 2, 0], [0, 1, 2]], [0, 0, 0]) - (1 / 2 + 1 / 3 + 1) / 3) < 1e-9
assert mrr([[1, 2]], [5]) == 0

CORPUS = [
    "The Lindenhof Observatory was founded in 1887 by the astronomer Clara Wendt.",
    "The main telescope of the Lindenhof Observatory is a 62-centimetre refractor named Greta.",
    "Visitors can tour the observatory on Thursday evenings; tickets cost 9 euros.",
    "The Varnholt Institute studies river sediments and is located on Mill Street in Lindenhof.",
    "Dr. Jonas Erken has directed the Varnholt Institute since 2019.",
    "The sediment archive of the institute holds 14,300 core samples stored at four degrees Celsius.",
    "Bus line 47 connects Lindenhof station with the observatory every twenty minutes.",
    "The Lindenhof town library closes at 6 pm on weekdays and stays closed on Sundays.",
    "The annual Lantern Festival in Lindenhof takes place on the second Saturday of October.",
    "The Sorrel Bridge, built from sandstone in 1754, is the oldest crossing over the river Alme.",
    "Error code LH-204 on the Greta telescope control unit means that the dome shutter failed to open.",
    "Error code LH-311 indicates that the tracking motor has overheated and must cool for thirty minutes.",
    "The bakery Kornblume on Market Square is known for its plum cake, sold only in September.",
    "The drinking water of Lindenhof comes from the Hallern reservoir, eleven kilometres north of the town.",
    "The sample submission form of the Varnholt Institute is called form S-17 and must be sent two weeks in advance.",
    "The Alme river floods most often in March, when snow melts in the Tessin hills.",
    "The football club of the town, SV Lindenhof 1921, plays its home matches at the Brunnen stadium.",
    "The Brunnen stadium has a capacity of 3,800 spectators.",
    "Mayor Ruth Albrecht was elected in 2022 for a term of six years.",
    "The weekly market is held on Wednesdays and Saturdays from 7 am to 1 pm on Market Square.",
    "The library of the observatory keeps the notebooks of Clara Wendt, which record 412 nights of observation.",
    "Parking near the observatory is limited to 30 spaces; the lot closes at midnight.",
    "The Tessin hills reach their highest point at the Rabenkopf, 847 metres above sea level.",
    "The Lindenhof museum of local history is housed in a former salt warehouse on Quay Lane.",
    "Error code LH-207 means that the focus motor has reached its end stop.",
    "Error code LH-240 means that the weather station reports wind above the safe limit.",
    "Error code LH-312 indicates that the tracking encoder has lost its reference position.",
    "Form S-71 of the Varnholt Institute is the request for access to the cold storage room.",
    "Form S-12 is used to report damaged sample containers to the archive staff.",
    "Bus line 74 runs from Lindenhof station to the Brunnen stadium on match days only.",
]
# (question, index of the passage that answers it, a string the right answer must contain)
QUERIES = [
    ("What does error code LH-204 mean?", 10, "shutter"),
    ("What is form S-17 used for?", 14, "submission"),
    ("Since which year has Jonas Erken directed the institute?", 4, "2019"),
    ("What is the capacity of the Brunnen stadium?", 17, "3,800"),
    ("How high is the Rabenkopf?", 22, "847"),
    ("What does LH-311 indicate?", 11, "overheat"),
    ("When was the Sorrel Bridge built?", 9, "1754"),
    ("Where does bus line 47 go?", 6, "observatory"),
    ("What does LH-207 mean?", 24, "focus"),
    ("What does error LH-312 indicate?", 26, "encoder"),
    ("What is form S-71 for?", 27, "cold storage"),
    ("Where does bus line 74 go?", 29, "stadium"),
    ("Who established the place where people look at the stars?", 0, "Wendt"),
    ("How much do I pay for an evening visit to see the telescope?", 2, "9 euros"),
    ("At what temperature are the drilled mud cylinders kept?", 5, "four"),
    ("In which month is the river most likely to overflow its banks?", 15, "March"),
    ("Where does the tap water of the town originate?", 13, "Hallern"),
    ("On which days can I buy fresh vegetables from stalls in the town centre?", 19, "Wednesday"),
    ("Who leads the local government?", 18, "Albrecht"),
    ("When is the yearly celebration with lights held?", 8, "October"),
]

NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME).eval()
bm, D = BM25(CORPUS), embed(model, tok, CORPUS)
gold = [g for _, g, _ in QUERIES]
runs = {"BM25": [], "embeddings": [], "RRF of both": []}
for q, _, _ in QUERIES:
    rb = ranking(bm.scores(q))
    rd = ranking(dense_scores(D, embed(model, tok, [q])[0]))
    runs["BM25"].append(rb); runs["embeddings"].append(rd); runs["RRF of both"].append(rrf([rb, rd]))

KEY, PARA = slice(0, 12), slice(12, 20)        # 12 questions that reuse the wording or an identifier, 8 paraphrases
m = {name: (mrr(r[KEY], gold[KEY]), mrr(r[PARA], gold[PARA]), recall_at_k(r, gold, 1), recall_at_k(r, gold, 3), mrr(r, gold)) for name, r in runs.items()}
assert m["BM25"][0] > 0.9 and m["BM25"][1] < 0.5, "keyword search: strong on shared wording, weak on paraphrases"
assert m["embeddings"][1] > 0.9, "embeddings: strong on paraphrases"
assert m["BM25"][0] >= m["embeddings"][0], "identifiers such as LH-204 and LH-207 are hard to tell apart for embeddings"
assert m["RRF of both"][0] >= m["embeddings"][0] and m["RRF of both"][1] > m["BM25"][1]

print("m72_hybrid: all tests passed")
print("                  MRR, shared wording   MRR, paraphrase   Recall@1   Recall@3   MRR, all 20")
for name, (mk, mp, r1, r3, ma) in m.items():
    print(f"  {name:14s} {mk:14.2f} {mp:18.2f} {r1:13.2f} {r3:10.2f} {ma:11.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

RRF 中名次从 1 开始计，用 `enumerate(r, start=1)`。

Ranks in RRF count from 1, so use `enumerate(r, start=1)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`r.index(g)` 给出正确段落在排序中的下标，名次是下标加 1。

`r.index(g)` gives the index of the correct passage in the ranking, and the rank is that index plus 1.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from collections import defaultdict


def ranking(scores):
    return sorted(range(len(scores)), key=lambda i: -scores[i])


def rrf(rankings, k=60):
    fused = defaultdict(float)
    for r in rankings:
        for position, doc in enumerate(r, start=1):
            fused[doc] += 1.0 / (k + position)
    return sorted(fused, key=lambda d: -fused[d])


def recall_at_k(rankings, gold, k):
    return sum(g in r[:k] for r, g in zip(rankings, gold)) / len(gold)


def mrr(rankings, gold):
    return sum(1.0 / (r.index(g) + 1) if g in r else 0.0 for r, g in zip(rankings, gold)) / len(gold)
```

</details>

## 3. 双编码器召回加交叉编码器重排 / Bi-encoder retrieval plus cross-encoder reranking

实现用交叉编码器给问题和段落的配对打分，并对前若干个候选重新排序。测试比较重排前后的 Recall@1、Recall@3 和 MRR，并分别统计第一阶段检索和重排所用的时间。

Implement scoring pairs of question and passage with a cross-encoder and reordering the first few candidates. The tests compare Recall@1, Recall@3 and MRR before and after reranking, and time the first-stage retrieval and the reranking separately.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# The embedding model of exercise 1 is a bi-encoder: question and passage are encoded separately and
# compared by a dot product, so passage vectors can be computed once and stored. A cross-encoder reads
# question and passage TOGETHER and outputs one relevance score. It is more accurate and far slower,
# so it is applied only to the best few candidates of a first, cheap search.


@torch.no_grad()
def cross_scores(model, tokenizer, query, passages):
    """Relevance score of every passage for the query, as a list of floats.

    Tokenise the pairs with tokenizer([query] * n, passages, padding=True, truncation=True, return_tensors="pt").
    The model returns logits of shape (n, 1); the score is that single number.
    """
    # TODO 1
    raise NotImplementedError


def rerank(model, tokenizer, query, corpus, candidates, top_n=8):
    """Re-order the first top_n candidates (document indices) by cross-encoder score, best first.
    Candidates after position top_n keep their order and follow unchanged."""
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import time
import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

CORPUS = [
    "The Lindenhof Observatory was founded in 1887 by the astronomer Clara Wendt.",
    "The main telescope of the Lindenhof Observatory is a 62-centimetre refractor named Greta.",
    "Visitors can tour the observatory on Thursday evenings; tickets cost 9 euros.",
    "The Varnholt Institute studies river sediments and is located on Mill Street in Lindenhof.",
    "Dr. Jonas Erken has directed the Varnholt Institute since 2019.",
    "The sediment archive of the institute holds 14,300 core samples stored at four degrees Celsius.",
    "Bus line 47 connects Lindenhof station with the observatory every twenty minutes.",
    "The Lindenhof town library closes at 6 pm on weekdays and stays closed on Sundays.",
    "The annual Lantern Festival in Lindenhof takes place on the second Saturday of October.",
    "The Sorrel Bridge, built from sandstone in 1754, is the oldest crossing over the river Alme.",
    "Error code LH-204 on the Greta telescope control unit means that the dome shutter failed to open.",
    "Error code LH-311 indicates that the tracking motor has overheated and must cool for thirty minutes.",
    "The bakery Kornblume on Market Square is known for its plum cake, sold only in September.",
    "The drinking water of Lindenhof comes from the Hallern reservoir, eleven kilometres north of the town.",
    "The sample submission form of the Varnholt Institute is called form S-17 and must be sent two weeks in advance.",
    "The Alme river floods most often in March, when snow melts in the Tessin hills.",
    "The football club of the town, SV Lindenhof 1921, plays its home matches at the Brunnen stadium.",
    "The Brunnen stadium has a capacity of 3,800 spectators.",
    "Mayor Ruth Albrecht was elected in 2022 for a term of six years.",
    "The weekly market is held on Wednesdays and Saturdays from 7 am to 1 pm on Market Square.",
    "The library of the observatory keeps the notebooks of Clara Wendt, which record 412 nights of observation.",
    "Parking near the observatory is limited to 30 spaces; the lot closes at midnight.",
    "The Tessin hills reach their highest point at the Rabenkopf, 847 metres above sea level.",
    "The Lindenhof museum of local history is housed in a former salt warehouse on Quay Lane.",
    "Error code LH-207 means that the focus motor has reached its end stop.",
    "Error code LH-240 means that the weather station reports wind above the safe limit.",
    "Error code LH-312 indicates that the tracking encoder has lost its reference position.",
    "Form S-71 of the Varnholt Institute is the request for access to the cold storage room.",
    "Form S-12 is used to report damaged sample containers to the archive staff.",
    "Bus line 74 runs from Lindenhof station to the Brunnen stadium on match days only.",
]
# (question, index of the passage that answers it, a string the right answer must contain)
QUERIES = [
    ("What does error code LH-204 mean?", 10, "shutter"),
    ("What is form S-17 used for?", 14, "submission"),
    ("Since which year has Jonas Erken directed the institute?", 4, "2019"),
    ("What is the capacity of the Brunnen stadium?", 17, "3,800"),
    ("How high is the Rabenkopf?", 22, "847"),
    ("What does LH-311 indicate?", 11, "overheat"),
    ("When was the Sorrel Bridge built?", 9, "1754"),
    ("Where does bus line 47 go?", 6, "observatory"),
    ("What does LH-207 mean?", 24, "focus"),
    ("What does error LH-312 indicate?", 26, "encoder"),
    ("What is form S-71 for?", 27, "cold storage"),
    ("Where does bus line 74 go?", 29, "stadium"),
    ("Who established the place where people look at the stars?", 0, "Wendt"),
    ("How much do I pay for an evening visit to see the telescope?", 2, "9 euros"),
    ("At what temperature are the drilled mud cylinders kept?", 5, "four"),
    ("In which month is the river most likely to overflow its banks?", 15, "March"),
    ("Where does the tap water of the town originate?", 13, "Hallern"),
    ("On which days can I buy fresh vegetables from stalls in the town centre?", 19, "Wednesday"),
    ("Who leads the local government?", 18, "Albrecht"),
    ("When is the yearly celebration with lights held?", 8, "October"),
]

CE = "cross-encoder/ms-marco-MiniLM-L-6-v2"                   # 23 million parameters, Apache 2.0
ce_tok = AutoTokenizer.from_pretrained(CE)
ce = AutoModelForSequenceClassification.from_pretrained(CE).eval()

s = cross_scores(ce, ce_tok, "How high is the Rabenkopf?", [CORPUS[22], CORPUS[12], CORPUS[17]])
assert len(s) == 3 and s[0] > s[1] + 3 and s[0] > s[2] + 3, "the passage that answers the question should score far above the others"
out = rerank(ce, ce_tok, "How high is the Rabenkopf?", CORPUS, [12, 17, 22, 5, 3], top_n=3)
assert out == [22] + [x for x in out[1:3]] + [5, 3] and set(out[:3]) == {12, 17, 22}, "only the first top_n are re-ordered"

NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME).eval()
bm, D = BM25(CORPUS), embed(model, tok, CORPUS)
gold = [g for _, g, _ in QUERIES]
dense, fused, dense_rr, fused_rr = [], [], [], []
t_first = t_rerank = 0.0
for q, _, _ in QUERIES:
    t = time.perf_counter()
    rd = ranking(dense_scores(D, embed(model, tok, [q])[0]))
    t_first += time.perf_counter() - t
    rf = rrf([ranking(bm.scores(q)), rd])
    t = time.perf_counter()
    dense_rr.append(rerank(ce, ce_tok, q, CORPUS, rd))
    t_rerank += time.perf_counter() - t
    dense.append(rd); fused.append(rf); fused_rr.append(rerank(ce, ce_tok, q, CORPUS, rf))

assert mrr(dense_rr, gold) >= mrr(dense, gold) and mrr(fused_rr, gold) > mrr(fused, gold) + 0.05
assert recall_at_k(dense_rr, gold, 1) >= 0.9

print("m72_rerank: all tests passed")
print("                               Recall@1   Recall@3    MRR")
for name, r in (("embeddings", dense), ("embeddings + reranking", dense_rr), ("RRF", fused), ("RRF + reranking", fused_rr)):
    print(f"  {name:26s} {recall_at_k(r, gold, 1):8.2f} {recall_at_k(r, gold, 3):10.2f} {mrr(r, gold):7.2f}")
print(f"  time for 20 questions: first search {t_first:.2f}s (passage vectors precomputed), reranking 8 candidates each {t_rerank:.2f}s")

<details><summary><b>提示 1 / Hint 1</b></summary>

成对输入的写法是 `tokenizer([query] * n, passages, padding=True, truncation=True, return_tensors='pt')`。

Pairs are passed as `tokenizer([query] * n, passages, padding=True, truncation=True, return_tensors='pt')`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

模型输出的 logits 形状是 `(n, 1)`，取 `[:, 0]`。

The logits of the model have shape `(n, 1)`. Take `[:, 0]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


@torch.no_grad()
def cross_scores(model, tokenizer, query, passages):
    enc = tokenizer([query] * len(passages), passages, padding=True, truncation=True, return_tensors="pt")
    return model(**enc).logits[:, 0].tolist()


def rerank(model, tokenizer, query, corpus, candidates, top_n=8):
    head, tail = candidates[:top_n], candidates[top_n:]
    scores = cross_scores(model, tokenizer, query, [corpus[i] for i in head])
    head = [doc for _, doc in sorted(zip(scores, head), key=lambda pair: -pair[0])]
    return head + tail
```

</details>

## 4. 基于自有文档的 mini-RAG / A mini-RAG over your own documents

把前三个练习的部件连接成完整的流程，即检索、融合、重排、拼接提示词、生成。测试在 20 个关于虚构小镇的问题上比较不使用检索和使用检索的正确率，并打印出检索命中而回答错误的例子。运行大约需要一分钟。完成后可以把 `CORPUS` 换成自己的文档，把问题换成自己的问题，流程不需要改动。

Connect the parts of the first three exercises into a full pipeline of retrieval, fusion, reranking, prompt assembly and generation. The tests compare accuracy without and with retrieval on 20 questions about an invented town and print the cases in which retrieval hit and the answer is wrong. The run takes about a minute. Afterwards `CORPUS` can be replaced by your own documents and the questions by your own, with no change to the pipeline.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

# Everything from exercises 1 to 3 is defined above: BM25, embed, dense_scores, ranking, rrf, rerank.

RAG_INSTRUCTION = ("Use only the following passages to answer the question. "
                   "If the answer is not in the passages, say \"I don't know\".")


def build_rag_prompt(question, passages):
    """The prompt that carries the retrieved passages. Exactly this layout:

    <RAG_INSTRUCTION>
    <blank line>
    [1] first passage
    [2] second passage
    <blank line>
    Question: <question>
    Answer in one short sentence.
    """
    # TODO 1
    raise NotImplementedError


@torch.no_grad()
def chat(model, tokenizer, user_message, max_new_tokens=40):
    """One user message in, the reply of the chat model out (greedy decoding, special tokens removed).

    enc = tokenizer.apply_chat_template([{"role": "user", "content": ...}], add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    Decode only the tokens generated after the prompt.
    """
    # TODO 2
    raise NotImplementedError


def retrieve(question, corpus, bm25, doc_vectors, embedder, embed_tok, cross, cross_tok, k=3):
    """The retrieval pipeline of exercises 1 to 3 in one function. Returns the indices of the k best passages.

    1. rank by embeddings   2. rank by BM25   3. fuse the two with rrf   4. rerank the fused list   5. keep k
    """
    # TODO 3
    raise NotImplementedError


def rag_answer(question, corpus, retriever, model, tokenizer):
    """Retrieve, build the prompt, generate. `retriever` is a function question -> list of passage indices.
    Returns (reply, ids)."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

p = build_rag_prompt("Who?", ["First.", "Second."])
assert p == RAG_INSTRUCTION + "\n\n[1] First.\n[2] Second.\n\nQuestion: Who?\nAnswer in one short sentence.", repr(p)

CORPUS = [
    "The Lindenhof Observatory was founded in 1887 by the astronomer Clara Wendt.",
    "The main telescope of the Lindenhof Observatory is a 62-centimetre refractor named Greta.",
    "Visitors can tour the observatory on Thursday evenings; tickets cost 9 euros.",
    "The Varnholt Institute studies river sediments and is located on Mill Street in Lindenhof.",
    "Dr. Jonas Erken has directed the Varnholt Institute since 2019.",
    "The sediment archive of the institute holds 14,300 core samples stored at four degrees Celsius.",
    "Bus line 47 connects Lindenhof station with the observatory every twenty minutes.",
    "The Lindenhof town library closes at 6 pm on weekdays and stays closed on Sundays.",
    "The annual Lantern Festival in Lindenhof takes place on the second Saturday of October.",
    "The Sorrel Bridge, built from sandstone in 1754, is the oldest crossing over the river Alme.",
    "Error code LH-204 on the Greta telescope control unit means that the dome shutter failed to open.",
    "Error code LH-311 indicates that the tracking motor has overheated and must cool for thirty minutes.",
    "The bakery Kornblume on Market Square is known for its plum cake, sold only in September.",
    "The drinking water of Lindenhof comes from the Hallern reservoir, eleven kilometres north of the town.",
    "The sample submission form of the Varnholt Institute is called form S-17 and must be sent two weeks in advance.",
    "The Alme river floods most often in March, when snow melts in the Tessin hills.",
    "The football club of the town, SV Lindenhof 1921, plays its home matches at the Brunnen stadium.",
    "The Brunnen stadium has a capacity of 3,800 spectators.",
    "Mayor Ruth Albrecht was elected in 2022 for a term of six years.",
    "The weekly market is held on Wednesdays and Saturdays from 7 am to 1 pm on Market Square.",
    "The library of the observatory keeps the notebooks of Clara Wendt, which record 412 nights of observation.",
    "Parking near the observatory is limited to 30 spaces; the lot closes at midnight.",
    "The Tessin hills reach their highest point at the Rabenkopf, 847 metres above sea level.",
    "The Lindenhof museum of local history is housed in a former salt warehouse on Quay Lane.",
    "Error code LH-207 means that the focus motor has reached its end stop.",
    "Error code LH-240 means that the weather station reports wind above the safe limit.",
    "Error code LH-312 indicates that the tracking encoder has lost its reference position.",
    "Form S-71 of the Varnholt Institute is the request for access to the cold storage room.",
    "Form S-12 is used to report damaged sample containers to the archive staff.",
    "Bus line 74 runs from Lindenhof station to the Brunnen stadium on match days only.",
]
# (question, index of the passage that answers it, a string the right answer must contain)
QUERIES = [
    ("What does error code LH-204 mean?", 10, "shutter"),
    ("What is form S-17 used for?", 14, "submission"),
    ("Since which year has Jonas Erken directed the institute?", 4, "2019"),
    ("What is the capacity of the Brunnen stadium?", 17, "3,800"),
    ("How high is the Rabenkopf?", 22, "847"),
    ("What does LH-311 indicate?", 11, "overheat"),
    ("When was the Sorrel Bridge built?", 9, "1754"),
    ("Where does bus line 47 go?", 6, "observatory"),
    ("What does LH-207 mean?", 24, "focus"),
    ("What does error LH-312 indicate?", 26, "encoder"),
    ("What is form S-71 for?", 27, "cold storage"),
    ("Where does bus line 74 go?", 29, "stadium"),
    ("Who established the place where people look at the stars?", 0, "Wendt"),
    ("How much do I pay for an evening visit to see the telescope?", 2, "9 euros"),
    ("At what temperature are the drilled mud cylinders kept?", 5, "four"),
    ("In which month is the river most likely to overflow its banks?", 15, "March"),
    ("Where does the tap water of the town originate?", 13, "Hallern"),
    ("On which days can I buy fresh vegetables from stalls in the town centre?", 19, "Wednesday"),
    ("Who leads the local government?", 18, "Albrecht"),
    ("When is the yearly celebration with lights held?", 8, "October"),
]

embed_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
embedder = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").eval()
cross_tok = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
cross = AutoModelForSequenceClassification.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2").eval()
LLM = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM).eval()

bm, D = BM25(CORPUS), embed(embedder, embed_tok, CORPUS)
retriever = lambda q: retrieve(q, CORPUS, bm, D, embedder, embed_tok, cross, cross_tok, k=3)
assert retriever("How high is the Rabenkopf?")[0] == 22 and len(retriever("How high is the Rabenkopf?")) == 3

reply = chat(llm, tok, "Reply with the single word: hello")
assert isinstance(reply, str) and "<|im_end|>" not in reply and "hello" in reply.lower()

# The corpus describes an invented town, so the model cannot know any of the answers from training.
correct = lambda reply, needle: needle.lower() in reply.lower()
rows, closed_ok, rag_ok, found = [], 0, 0, 0
for q, gold, needle in QUERIES:
    closed = chat(llm, tok, q + " Answer in one short sentence.")
    reply, ids = rag_answer(q, CORPUS, retriever, llm, tok)
    closed_ok += correct(closed, needle); rag_ok += correct(reply, needle); found += gold in ids
    rows.append((q, gold in ids, correct(reply, needle), closed, reply))
assert closed_ok <= 4, "without the passages the model can only guess"
assert found >= 17 and rag_ok >= 12, (found, rag_ok)

# With passages that do not contain the answer, the instruction asks for "I don't know".
off_topic = chat(llm, tok, build_rag_prompt("What is the capacity of the Brunnen stadium?", [CORPUS[12], CORPUS[7]]))

print(f"m72_minirag: all tests passed")
print(f"  20 questions about an invented town: {closed_ok} answered correctly without retrieval, {rag_ok} with retrieval")
print(f"  the passage with the answer was among the 3 retrieved for {found} questions")
print("  retrieved the right passage and still answered wrongly:")
for q, hit, ok, closed, reply in rows:
    if hit and not ok:
        print(f"    Q: {q}\n       {reply!r}")
print("  an example without retrieval:", repr(rows[3][3]))
print("  the same question with retrieval:", repr(rows[3][4]))
print("  passages without the answer:", repr(off_topic))

<details><summary><b>提示 1 / Hint 1</b></summary>

段落编号从 1 开始，用 `enumerate(passages, start=1)`，每行的格式是 `[1] 段落`。

Passages are numbered from 1 with `enumerate(passages, start=1)`, each line in the form `[1] passage`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`retrieve` 的五步对应五行代码，前几个练习的函数可以直接调用。

The five steps of `retrieve` correspond to five lines of code, and the functions of the earlier exercises can be called directly.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`chat` 中用 `return_dict=True` 得到 `input_ids` 和 `attention_mask`，再用 `model.generate(**enc, ...)`。

In `chat` use `return_dict=True` to get `input_ids` and `attention_mask`, then `model.generate(**enc, ...)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

RAG_INSTRUCTION = ("Use only the following passages to answer the question. "
                   "If the answer is not in the passages, say \"I don't know\".")


def build_rag_prompt(question, passages):
    numbered = "\n".join(f"[{i}] {p}" for i, p in enumerate(passages, start=1))
    return f"{RAG_INSTRUCTION}\n\n{numbered}\n\nQuestion: {question}\nAnswer in one short sentence."


@torch.no_grad()
def chat(model, tokenizer, user_message, max_new_tokens=40):
    enc = tokenizer.apply_chat_template([{"role": "user", "content": user_message}], add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def retrieve(question, corpus, bm25, doc_vectors, embedder, embed_tok, cross, cross_tok, k=3):
    first = ranking(dense_scores(doc_vectors, embed(embedder, embed_tok, [question])[0]))
    fused = rrf([ranking(bm25.scores(question)), first])
    return rerank(cross, cross_tok, question, corpus, fused)[:k]


def rag_answer(question, corpus, retriever, model, tokenizer):
    ids = retriever(question)
    reply = chat(model, tokenizer, build_rag_prompt(question, [corpus[i] for i in ids]))
    return reply, ids
```

</details>

## 5. 分块：固定窗口与按句子打包 / Chunking: fixed windows and sentence packing

建立索引之前，长文档要先切成块。实现两种切法。`chunk_words(text, size, overlap)` 按空格分词，每块 `size` 个词，相邻两块共享 `overlap` 个词，最后一块可以较短，但不应出现完全被前一块包含的多余的块。`chunk_sentences(text, max_words)` 以句子为单位，把连续的句子装进同一块，直到再加一句就会超过 `max_words`，单独一句超过上限时自成一块。测试的最后一部分用一段文字说明重叠的作用。一条事实的两半落在块的边界两侧时，没有重叠的切法得不到同时包含两半的块，有重叠时可以。

Before an index is built, long documents are cut into chunks. Implement two ways of cutting. `chunk_words(text, size, overlap)` splits on whitespace, puts `size` words into each chunk and lets neighbouring chunks share `overlap` words. The last chunk may be shorter, but no extra chunk that is fully contained in the previous one should appear. `chunk_sentences(text, max_words)` works on sentences and packs consecutive sentences into one chunk until one more would exceed `max_words`, with a single sentence above the limit forming a chunk of its own. The last part of the tests uses a passage to show what overlap is for. When the two halves of a fact fall on either side of a chunk boundary, cutting without overlap yields no chunk that holds both halves, and cutting with overlap does.

In [ ]:
import re

def chunk_words(text, size, overlap=0):
    """Windows of `size` words that share `overlap` words with their neighbour. Returns a list of strings."""
    assert 0 <= overlap < size
    words = text.split()
    # TODO: slide a window of `size` words forward by size - overlap, stop once the end is covered
    raise NotImplementedError

def chunk_sentences(text, max_words):
    """Pack whole sentences into chunks of at most `max_words` words. Returns a list of strings."""
    # TODO: split into sentences, then add sentences to the current chunk while they fit
    raise NotImplementedError

In [ ]:
# 检查 / checks
text = " ".join(f"w{i}" for i in range(10))
assert chunk_words(text, 4) == ["w0 w1 w2 w3", "w4 w5 w6 w7", "w8 w9"]
assert chunk_words(text, 4, 2) == ["w0 w1 w2 w3", "w2 w3 w4 w5", "w4 w5 w6 w7", "w6 w7 w8 w9"], "no trailing chunk made of overlap only"
assert chunk_words(text, 5, 1) == ["w0 w1 w2 w3 w4", "w4 w5 w6 w7 w8", "w8 w9"]
assert chunk_words(text, 20, 5) == [text] and chunk_words("", 4) == []
for size, ov in ((3, 0), (4, 1), (6, 3), (7, 2)):
    cs = chunk_words(text, size, ov)
    assert all(len(c.split()) <= size for c in cs)
    assert [w for i, c in enumerate(cs) for w in c.split()[(ov if i else 0):]] == text.split(), "every word is covered once outside the overlaps"

s = "One two three. Four five! Six seven eight nine? Ten."
assert chunk_sentences(s, 5) == ["One two three. Four five!", "Six seven eight nine? Ten."]
assert chunk_sentences(s, 4) == ["One two three.", "Four five!", "Six seven eight nine?", "Ten."]
assert chunk_sentences(s, 3) == ["One two three.", "Four five!", "Six seven eight nine?", "Ten."], "a sentence above the limit stands alone"
assert chunk_sentences(s, 100) == [s]

# What overlap is for: the name and the year of one fact sit on either side of a boundary.
doc = ("The Lindenhof observatory stands on the northern hill above the old town and has been open to visitors "
       "for many decades. Its main telescope was built by the instrument maker Clara Voss and it was installed "
       "in 1931 after a long delay caused by a harsh winter. The dome was renovated twice since then.")
both = lambda chunks: [c for c in chunks if "Voss" in c and "1931" in c]
assert len(both(chunk_words(doc, 34, 0))) == 0, "without overlap the boundary separates the two halves"
assert len(both(chunk_words(doc, 34, 8))) >= 1, "with overlap one chunk holds both"
assert len(both(chunk_sentences(doc, 34))) == 1, "cutting at sentence boundaries keeps the sentence whole"
print("ok")

<details><summary><b>提示 1 / Hint 1</b></summary>

窗口每次前进 `size - overlap` 个词。起点为 `i` 的窗口覆盖到 `i + size`，当它已经到达末尾时就应当停止，否则会多出一个只含重叠部分的块。

The window advances by `size - overlap` words. A window starting at `i` covers up to `i + size`, and the loop should stop once it has reached the end, since one more chunk holding only the overlap would otherwise appear.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

句子可以用 `re.split(r'(?<=[.!?])\s+', text.strip())` 切开。打包时维护当前块的句子列表和词数。

Sentences can be split with `re.split(r'(?<=[.!?])\s+', text.strip())`. While packing, keep the list of sentences of the current chunk and its word count.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import re

def chunk_words(text, size, overlap=0):
    """Windows of `size` words that share `overlap` words with their neighbour. Returns a list of strings."""
    assert 0 <= overlap < size
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + size]))
        if i + size >= len(words):
            break
        i += size - overlap
    return chunks

def chunk_sentences(text, max_words):
    """Pack whole sentences into chunks of at most `max_words` words. Returns a list of strings."""
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    chunks, current, count = [], [], 0
    for s in sentences:
        n = len(s.split())
        if current and count + n > max_words:
            chunks.append(" ".join(current)); current, count = [], 0
        current.append(s); count += n
    if current:
        chunks.append(" ".join(current))
    return chunks
```

</details>

## 6. 倒排文件索引：用少量比较换取近似结果 / An inverted file index: approximate results from few comparisons

精确检索要把问题与每一个向量比较。倒排文件索引（IVF）先用 k-means 把向量分成 `nlist` 个簇，检索时只在与问题最接近的 `nprobe` 个簇里查找。实现 `IVFIndex` 的 `add` 和 `search`，`search` 返回 `(ids, n_compared)`，其中 `n_compared` 是比较过的向量个数，等于 `nlist` 个质心加上被扫描的簇里的向量数。数据为 4096 个 32 维的单位向量，`nlist` 取 256，即 Faiss 文档对 100 万以下的数据给出的范围 $4\sqrt{N}$ 至 $16\sqrt{N}$ 的下限。测试在两组合成数据上测量召回率，一组有明显的簇结构，一组没有，以说明近似检索的效果取决于数据的分布。安装了 `faiss-cpu` 时，测试还会把精确检索的结果与 Faiss 的 `IndexFlatIP` 比较。

Exact search compares the query with every vector. An inverted file index (IVF) first groups the vectors into `nlist` clusters with k-means and at query time looks only into the `nprobe` clusters closest to the query. Implement `add` and `search` of `IVFIndex`. `search` returns `(ids, n_compared)`, where `n_compared` is the number of vectors compared, equal to the `nlist` centroids plus the vectors in the scanned clusters. The data are 4096 unit vectors of 32 dimensions and `nlist` is 256, the lower end of the range $4\sqrt{N}$ to $16\sqrt{N}$ that the Faiss documentation gives for data below one million vectors. The tests measure recall on two synthetic data sets, one with a clear cluster structure and one without, to show that the quality of approximate search depends on the distribution of the data. With `faiss-cpu` installed, the tests also compare exact search with `IndexFlatIP` of Faiss.

In [ ]:
import numpy as np

def make_data(n=4096, d=32, topics=64, n_queries=200, seed=0):
    """Unit vectors scattered around `topics` centres. topics == n gives data without cluster structure."""
    rng = np.random.default_rng(seed)
    centres = rng.normal(size=(topics, d))
    x = centres[rng.integers(topics, size=n)] + 0.7 * rng.normal(size=(n, d))
    q = centres[rng.integers(topics, size=n_queries)] + 0.7 * rng.normal(size=(n_queries, d))
    unit = lambda a: (a / np.linalg.norm(a, axis=1, keepdims=True)).astype(np.float32)
    return unit(x), unit(q)

def kmeans(x, k, iters=10, seed=0):
    """Spherical k-means: assign by largest inner product, renormalize the centroids."""
    rng = np.random.default_rng(seed)
    c = x[rng.choice(len(x), k, replace=False)].copy()
    for _ in range(iters):
        assign = np.argmax(x @ c.T, axis=1)
        for j in range(k):
            members = x[assign == j]
            if len(members):
                c[j] = members.mean(0)
        c /= np.linalg.norm(c, axis=1, keepdims=True)
    return c

def flat_search(x, q, k):
    """Exact search: compare the query with every vector. Returns the ids of the k largest inner products."""
    return np.argsort(-(x @ q))[:k]


class IVFIndex:
    def __init__(self, centroids):
        self.centroids = centroids          # (nlist, d)
        self.lists = None                   # one array of vector ids per cluster
        self.x = None

    def add(self, x):
        """Store the vectors and the ids that fall into each cluster."""
        # TODO: assign every vector to its closest centroid and fill self.lists
        raise NotImplementedError

    def search(self, q, k, nprobe):
        """Return (ids of the k best vectors among the nprobe closest clusters, number of vectors compared)."""
        # TODO: rank the centroids, collect the ids of the nprobe best clusters, search exactly among them
        raise NotImplementedError

In [ ]:
# 检查 / checks
def evaluate(topics):
    x, queries = make_data(topics=topics)
    index = IVFIndex(kmeans(x, 256)); index.add(x)
    out = {}
    for nprobe in (1, 4, 16, 64, 256):
        recall, compared = 0.0, 0
        for q in queries:
            ids, n = index.search(q, 10, nprobe)
            recall += len(set(ids.tolist()) & set(flat_search(x, q, 10).tolist())) / 10
            compared += n
        out[nprobe] = (recall / len(queries), compared / len(queries) / len(x))
    return x, queries, index, out

x, queries, index, clustered = evaluate(topics=64)
assert sum(len(l) for l in index.lists) == len(x) and len(index.lists) == 256
ids, n = index.search(queries[0], 10, 256)
assert n == len(x) + 256 and set(ids.tolist()) == set(flat_search(x, queries[0], 10).tolist()), "probing every cluster is exact search"
ids, n = index.search(queries[0], 10, 1)
assert n < 0.2 * len(x) and len(ids) <= 10

print("clustered data    nprobe -> (recall@10, share of vectors compared)")
for k, (r, c) in clustered.items(): print(f"  {k:>3}  {r:.3f}  {c:.3f}")
_, _, _, uniform = evaluate(topics=4096)
print("data without cluster structure")
for k, (r, c) in uniform.items(): print(f"  {k:>3}  {r:.3f}  {c:.3f}")

r = [clustered[k][0] for k in (1, 4, 16, 64, 256)]
assert all(a <= b + 1e-9 for a, b in zip(r, r[1:])) and r[-1] == 1.0, "recall grows with nprobe"
assert clustered[4][0] > 0.9 and clustered[4][1] < 0.15, "on clustered data a few probes are enough"
assert uniform[4][0] < 0.6 and uniform[64][0] > 0.8, "without cluster structure far more probes are needed"

try:
    import faiss
    flat = faiss.IndexFlatIP(x.shape[1]); flat.add(x)
    _, I = flat.search(queries[:20], 10)
    assert all(set(I[i].tolist()) == set(flat_search(x, queries[i], 10).tolist()) for i in range(20))
    print("exact search agrees with faiss.IndexFlatIP")
except ImportError:
    print("faiss is not installed, the cross-check was skipped")
print("ok")

<details><summary><b>提示 1 / Hint 1</b></summary>

`add` 中先求每个向量所属的簇，`np.argmax(x @ self.centroids.T, axis=1)`，再为每个簇保存属于它的向量编号。

In `add`, first find the cluster of every vector with `np.argmax(x @ self.centroids.T, axis=1)`, then store for every cluster the ids of its vectors.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`search` 中先把问题与全部质心比较，取内积最大的 `nprobe` 个簇，把这些簇的编号拼起来，再只在这些向量中做精确检索。

In `search`, compare the query with all centroids first, take the `nprobe` clusters with the largest inner product, concatenate their ids and run exact search among these vectors alone.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import numpy as np

def make_data(n=4096, d=32, topics=64, n_queries=200, seed=0):
    """Unit vectors scattered around `topics` centres. topics == n gives data without cluster structure."""
    rng = np.random.default_rng(seed)
    centres = rng.normal(size=(topics, d))
    x = centres[rng.integers(topics, size=n)] + 0.7 * rng.normal(size=(n, d))
    q = centres[rng.integers(topics, size=n_queries)] + 0.7 * rng.normal(size=(n_queries, d))
    unit = lambda a: (a / np.linalg.norm(a, axis=1, keepdims=True)).astype(np.float32)
    return unit(x), unit(q)

def kmeans(x, k, iters=10, seed=0):
    """Spherical k-means: assign by largest inner product, renormalize the centroids."""
    rng = np.random.default_rng(seed)
    c = x[rng.choice(len(x), k, replace=False)].copy()
    for _ in range(iters):
        assign = np.argmax(x @ c.T, axis=1)
        for j in range(k):
            members = x[assign == j]
            if len(members):
                c[j] = members.mean(0)
        c /= np.linalg.norm(c, axis=1, keepdims=True)
    return c

def flat_search(x, q, k):
    """Exact search: compare the query with every vector. Returns the ids of the k largest inner products."""
    return np.argsort(-(x @ q))[:k]


class IVFIndex:
    def __init__(self, centroids):
        self.centroids = centroids          # (nlist, d)
        self.lists = None                   # one array of vector ids per cluster
        self.x = None

    def add(self, x):
        """Store the vectors and the ids that fall into each cluster."""
        self.x = x
        assign = np.argmax(x @ self.centroids.T, axis=1)
        self.lists = [np.where(assign == j)[0] for j in range(len(self.centroids))]

    def search(self, q, k, nprobe):
        """Return (ids of the k best vectors among the nprobe closest clusters, number of vectors compared)."""
        cells = np.argsort(-(self.centroids @ q))[:nprobe]
        ids = np.concatenate([self.lists[j] for j in cells])
        best = ids[np.argsort(-(self.x[ids] @ q))[:k]]
        return best, len(self.centroids) + len(ids)
```

</details>